In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (StandardScaler, OneHotEncoder, 
                                  OrdinalEncoder, LabelEncoder)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             ConfusionMatrixDisplay, RocCurveDisplay,
                             classification_report)
from sklearn.impute import SimpleImputer

# Import classification models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, 
                              GradientBoostingClassifier,
                              HistGradientBoostingClassifier)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.neural_network import MLPClassifier

# 1. Load your dataset
# Replace with your actual file path and target column name
df= pd.read_csv('C:/Users/Hp Laptop/Desktop/data/unass.csv')
target_column = 'unassist'

# 2. Preprocessing setup
# Identify categorical and numerical features
categorical_features = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
if target_column in categorical_features:
    categorical_features.remove(target_column)
    
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if target_column in numerical_features:
    numerical_features.remove(target_column)

# Preprocessing pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 3. Initialize models with handling for categorical features
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "SVM": SVC(probability=True, class_weight='balanced'),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(class_weight='balanced'),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(),
    "CatBoost": CatBoostClassifier(verbose=0),
    "HistGradientBoosting": HistGradientBoostingClassifier(),
    "CategoricalNB": CategoricalNB(),
    "Neural Network": MLPClassifier(max_iter=1000, early_stopping=True)
}

# 4. Data preparation
X = df.drop(columns=[target_column])
y = df[target_column]

# Encode target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 5. Train and evaluate models
results = []
for name, model in models.items():
    # Create pipeline with preprocessing and model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Train model
    pipeline.fit(X_train, y_train)
    
    # Predictions
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]  # Probabilities for ROC-AUC
    
    # Calculate metrics
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }
    results.append(metrics)
    
    # Print classification report
    print(f"\n{name} Classification Report:")
    print(classification_report(y_test, y_pred))
    
    # Generate plots
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'{name} Performance', fontsize=16)
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm).plot(ax=ax[0])
    ax[0].set_title('Confusion Matrix')
    
    # ROC Curve
    RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax[1])
    ax[1].set_title('ROC Curve')
    plt.tight_layout()
    plt.show()

# 6. Results comparison
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print("\n=== Model Performance Comparison ===")
print(results_df.round(3))

# 7. Feature importance for tree-based models
plt.figure(figsize=(15, 10))
for i, (name, model) in enumerate(models.items()):
    if hasattr(model, 'feature_importances_'):
        # Get feature names after preprocessing
        try:
            # Get feature names from one-hot encoder
            ohe_features = pipeline.named_steps['preprocessor']\
                .named_transformers_['cat']\
                .named_steps['encoder']\
                .get_feature_names_out(categorical_features)
            
            all_features = numerical_features + list(ohe_features)
            
            importances = pipeline.named_steps['classifier'].feature_importances_
            sorted_idx = importances.argsort()[-15:]  # Top 15 features
            
            plt.subplot(4, 3, i+1)
            plt.barh(range(len(sorted_idx)), importances[sorted_idx])
            plt.yticks(range(len(sorted_idx)), np.array(all_features)[sorted_idx])
            plt.title(f"{name} Feature Importances")
        except AttributeError:
            continue
plt.tight_layout()
plt.show()